# Layer placement and needling operations

These recipe commands approximate manufacturing by holding fibers on
temporary device-resident targets. Each hold must be followed by an
explicit relaxation. Releasing a target lets the displaced fibers and
their contacts settle mechanically.

Placement and needling return a `HeldTargets` handle. Use it as a
context manager: the targets are held for the relaxation inside the
`with` block and released when the block ends. Without `with`, call
`release_layer_placement()` or `release_needles()` yourself.

In [ ]:
import tangle
from tangle.units import mm, um

# periodic="xy" leaves z as the only bounded axis, so z is the stack
# axis for layer and needle motion.
recipe = tangle.Recipe(tangle.Cell([1 * mm, 1 * mm, 3 * mm], periodic="xy"))
recipe.stack_axis

## Layer commands

- `scale_layer_spacing(factor, stiffness=, max_translation=)` scales
  all current layer-center spacings.
- `place_layer_above(layer, gap=, ...)` brings one layer to a surface
  gap above the active stack.
- `settle_targets(tolerance=, max_iterations=)` relaxes until held
  fibers reach their targets.
- `release_layer_placement()` removes those temporary targets; a `with`
  block calls it for you.
- `fit_cell_to_active_fibers(axes=, padding=)` removes empty domain
  space; `axes` accepts letters such as `"z"`.

In [ ]:
# scale_layer_spacing scales current layer-center spacing; it does not
# teleport fibers or bypass contact relaxation. The with block
# releases the layer targets when it ends.
with recipe.scale_layer_spacing(0.8, stiffness=0.5, max_translation=5 * um):
    recipe.relax_for(500)

# Without `with`, the targets stay held until released explicitly.
recipe.place_layer_above(2, gap=2 * um, stiffness=0.5, max_translation=5 * um)
recipe.settle_targets(tolerance=0.1 * um, max_iterations=2_000)
recipe.release_layer_placement()

# Shrink the bounded z extent to the active fibers plus padding.
recipe.fit_cell_to_active_fibers(axes="z", padding=25 * um)

## Needle commands

`needle_layer(layer, footprint=, depth=, ...)` pulls one internal
vertex of each eligible fiber in `layer` along the stack axis by
`depth`. The footprint chooses the fibers:

- `CircularFootprint(center, diameter=)` selects fibers crossing an
  in-plane circle; `center` holds the two in-plane coordinates.
- `CircularFootprint.random(diameter=, seed=)` places that circle at a
  seeded random center. The layer index is mixed into the seed, so one
  seed gives different punch locations on different layers.
- `RandomFiberFraction(fraction, seed=)` samples a fraction of the
  layer's eligible fibers regardless of location.

`min_fiber_diameter` restricts needling to thick fibers; `stiffness`,
`max_translation`, and `max_translation_over_diameter` limit how fast
the targets pull.

In [ ]:
# A fixed circular punch in x/y (the axes other than the stack axis).
with recipe.needle_layer(
    2,
    footprint=tangle.CircularFootprint([0.45 * mm, 0.55 * mm], diameter=100 * um),
    depth=0.6 * mm,
    min_fiber_diameter=15 * um,
    stiffness=0.75,
    max_translation=5 * um,
    max_translation_over_diameter=0.5,
):
    # Hold the target through relaxation; the block end releases it.
    recipe.settle_targets(tolerance=0.1 * um, max_iterations=5_000)

# Seeded random punch locations: one stroke per seed.
for stroke in range(3):
    with recipe.needle_layer(
        2,
        footprint=tangle.CircularFootprint.random(diameter=150 * um, seed=stroke),
        depth=350 * um,
    ):
        recipe.settle_targets(tolerance=0.1 * um, max_iterations=2_000)

# Random needling samples fibers reproducibly by fraction rather than
# by spatial footprint. Here the needle is released explicitly.
recipe.needle_layer(
    3,
    footprint=tangle.RandomFiberFraction(0.15, seed=2026),
    depth=0.6 * mm,
    min_fiber_diameter=15 * um,
    stiffness=0.75,
    max_translation=5 * um,
    max_translation_over_diameter=0.5,
)
recipe.settle_targets(tolerance=0.1 * um, max_iterations=5_000)
recipe.release_needles()
print(*recipe.operations(), sep="\n")